In [5]:
from document_qa_chatbot.ingestion import (
    extract_text_from_pdfs,
    chunk_documents
)

from pathlib import Path

# PDF file paths
pdf_paths = [
    Path("data/Investor Awareness Booklet (English).pdf"),
    Path("data/12 NISM-Series-VIII Equity Derivatives Certification Examination.pdf")
]

# Check PDF files
for pdf_path in pdf_paths:
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

# Extract text
documents = extract_text_from_pdfs(pdf_paths)

# Create chunks
chunks = chunk_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 2333


In [6]:
# Display first five chunks
for chunk in chunks[:5]:
    print("\nChunk ID:", chunk["chunk_id"])
    print("Document:", chunk["document_name"])
    print("Page:", chunk["page_number"])
    print("Text:", chunk["text"][:200])

# Count chunks per document
print("\n--- DOCUMENT CHUNK COUNTS ---")

for document_name in sorted({
    chunk["document_name"] for chunk in chunks
}):
    count = sum(
        1 for chunk in chunks
        if chunk["document_name"] == document_name
    )

    print(document_name, ":", count, "chunks")

# Validate chunk IDs
chunk_ids = [chunk["chunk_id"] for chunk in chunks]

print("\n--- CHUNK ID VALIDATION ---")
print("Total IDs:", len(chunk_ids))
print("Unique IDs:", len(set(chunk_ids)))

if len(chunk_ids) == len(set(chunk_ids)):
    print("All chunk IDs are unique!")
else:
    print("Duplicate chunk IDs found!")


Chunk ID: e4fe30bc9fca10f10ebbb59283d651ec
Document: Investor Awareness Booklet (English).pdf
Page: 1
Text: THE INVESTOR 
AWARENESS 
WORKBOOK 
UNDERSTANDING INDIA’S 
SECURITIES MARKET 
Partners

Chunk ID: 99f2a9e4650c0840b2e612f67e89000f
Document: Investor Awareness Booklet (English).pdf
Page: 2
Text: The Investor Awareness Workbook | 1

Chunk ID: 5825ba2a075fba330f7fb88f53713d05
Document: Investor Awareness Booklet (English).pdf
Page: 3
Text: 2 | The Investor Awareness Workbook 
 
 
ACKNOWLEDGEMENT 
This Investor Awareness Workbook has been prepared under Project 
Jagrook, which has been initiated by the Securities and Exchange Board 
of I

Chunk ID: c2a21c395eed384ba1874a9e2fdd6f41
Document: Investor Awareness Booklet (English).pdf
Page: 3
Text: approach to promote financial literacy, informed investment practices, 
and prevention of financial fraud. 
This unified mission is being driven by SEBI in collaboration with key 
Securities Market Pa

Chunk ID: c347be11a43a8ebfb6cf12446ae4f

In [7]:
questions = {
    "Q1": {
        "question": "What is a mutual fund?",
        "keywords": ["mutual fund", "pool of money", "investors"]
    },
    "Q2": {
        "question": "How do I choose a mutual fund?",
        "keywords": ["choose a fund", "selection process", "investment goal", "fund factsheet"]
    },
    "Q3": {
        "question": "What is SIP, and how does it work?",
        "keywords": ["systematic investment plan", "SIP", "fixed amount"]
    },
    "Q4": {
        "question": "What is the difference between equity funds and debt funds?",
        "keywords": ["equity funds", "debt funds", "fixed-income instruments"]
    },
    "Q5": {
        "question": "What is a Risk-o-meter, and how does it help investors understand mutual fund risk?",
        "keywords": ["Risk-o-meter", "risk level", "risk assessment"]
    },
    "Q6": {
        "question": "What is the difference between direct and regular mutual fund plans?",
        "keywords": ["direct plan", "regular plan", "distribution commission"]
    },
    "Q7": {
        "question": "What is NAV (Net Asset Value), and how is it calculated?",
        "keywords": ["Net Asset Value", "NAV", "units outstanding"]
    },
    "Q8": {
        "question": "What is the difference between the primary market and the secondary market?",
        "keywords": ["primary market", "secondary market"]
    },
    "Q9": {
        "question": "What is an IPO, and how does it work?",
        "keywords": ["Initial Public Offering", "IPO", "public issue"]
    },
    "Q10": {
        "question": "What is a demat account, and why is it required?",
        "keywords": ["demat account", "dematerialised", "electronic form"]
    }
}

print("Total evaluation questions:", len(questions))

Total evaluation questions: 10


In [8]:
print("\n========== BATCH GROUND-TRUTH SEARCH ==========")

for q_id, details in questions.items():

    print(f"\n{'=' * 70}")
    print(f"{q_id}: {details['question']}")
    print("=" * 70)

    matches = []

    for chunk in chunks:
        text = chunk["text"].lower()

        if any(
            keyword.lower() in text
            for keyword in details["keywords"]
        ):
            matches.append(chunk)

    print(f"Matching chunks: {len(matches)}")

    for chunk in matches[:10]:
        print(f"\nChunk ID: {chunk['chunk_id']}")
        print(f"Document: {chunk['document_name']}")
        print(f"Page: {chunk['page_number']}")
        print(f"Text: {chunk['text'][:700]}")


========== BATCH GROUND-TRUTH SEARCH ==========

Q1: What is a mutual fund?
Matching chunks: 275

Chunk ID: c2a21c395eed384ba1874a9e2fdd6f41
Document: Investor Awareness Booklet (English).pdf
Page: 3
Text: approach to promote financial literacy, informed investment practices, 
and prevention of financial fraud. 
This unified mission is being driven by SEBI in collaboration with key 
Securities Market Partners (SMPs), including 
• Association of Mutual Funds in India (AMFI) 
• National Stock Exchange of India Limited (NSE) 
• BSE Limited (BSE) 
• Multi Commodity Exchange of India Limited (MCX) 
• Central Depository Services (India) Limited (CDSL) 
• National Securities Depository Limited (NSDL)

Chunk ID: c347be11a43a8ebfb6cf12446ae4ff13
Document: Investor Awareness Booklet (English).pdf
Page: 3
Text: • National Securities Depository Limited (NSDL) 
• National Institute of Securities Markets (NISM) 
The content of this Workbook has been developed and curated by NISM. 
NISM gratefully a

In [9]:
ground_truth = {
    "Q1": {
        "question": "What is a mutual fund?",
        "relevant_chunk_ids": [
            "f2b276f9d32082802223f3d4eaf28234"
        ]
    },

    "Q2": {
        "question": "How do I choose a mutual fund?",
        "relevant_chunk_ids": [
            "54fcbd387ffb7b8256e048b0a7758077",
            "24f67f008628cb5a7df574645e77938a",
            "46286916f98b3212e7d506ae7dc528e1",
            "a9d76003d839997fa29fad3fa59d29d5",
            "7d6bb1dfcecc63108956b65382cb412e"
        ]
    },

    "Q3": {
        "question": "What is SIP, and how does it work?",
        "relevant_chunk_ids": [
            "6e348284b1e9990817d65073177b922a",
            "7dc7ed336962edcf078b4401c48abf9f",
            "cff7149e5fc5a1998b88d2d25f694706",
            "986c56696524ec9d9c9aa7c4ea0ea83a"
        ]
    },

    "Q4": {
        "question": "What is the difference between equity funds and debt funds?",
        "relevant_chunk_ids": [
            "10dfb532ca7875a5bafd9498b56a257e",
            "22fb4b234f74750b7fcb5ac9174a85db",
            "6df4ec1757bb4e82c18fb6bbfaf2763d"
        ]
    },

    "Q5": {
        "question": "What is a Risk-o-meter?",
        "relevant_chunk_ids": [
            "e08e6b9d7909ae2b42e3094c64f58d25",
            "087cd1c7124e408d5826c124d99f08bc",
            "a9d76003d839997fa29fad3fa59d29d5"
        ]
    },

    "Q6": {
        "question": "What is the difference between direct and regular mutual fund plans?",
        "relevant_chunk_ids": [
            "2e5d4e65021661d42c0580c03acb9675",
            "4dd9a36b1713547b0d40cfd64565f8b9"
        ]
    },

    "Q7": {
        "question": "What is NAV?",
        "relevant_chunk_ids": [
            "d65963531ccb47fee68d4a4f1f555712"
        ]
    },

    "Q8": {
        "question": "What is the difference between primary and secondary markets?",
        "relevant_chunk_ids": [
            "a5e4c6d4c4caabee8d86f14532e1f781",
            "4e758f064c6edd0f2db9b367438b43de"
        ]
    },

    "Q9": {
        "question": "What is an IPO, and how does it work?",
        "relevant_chunk_ids": [
            "0de8f02cf6029c02e768ae5f7724c99e",
            "d053ee41dc291dd3b67dc923ebd4a8c6",
            "d0cd87bb8b7e08e6e07ac4861ab00fa4"
        ]
    },

    "Q10": {
        "question": "What is a demat account, and why is it required?",
        "relevant_chunk_ids": [
            "f5dca177aa46ad2ea85006e13ebcab15",
            "88bf4f583d3de4943ce4b5a9ec3f6b2d",
            "a4a00ab53a84b86ab8a35df150839bb8"
        ]
    }
}

In [10]:
available_ids = {
    chunk["chunk_id"]
    for chunk in chunks
}

for question_id, data in ground_truth.items():

    valid_ids = []
    invalid_ids = []

    for chunk_id in data["relevant_chunk_ids"]:

        if chunk_id in available_ids:
            valid_ids.append(chunk_id)
        else:
            invalid_ids.append(chunk_id)

    print(f"\n{question_id}: {data['question']}")
    print("Valid IDs:", len(valid_ids))
    print("Invalid IDs:", invalid_ids)


Q1: What is a mutual fund?
Valid IDs: 1
Invalid IDs: []

Q2: How do I choose a mutual fund?
Valid IDs: 5
Invalid IDs: []

Q3: What is SIP, and how does it work?
Valid IDs: 4
Invalid IDs: []

Q4: What is the difference between equity funds and debt funds?
Valid IDs: 3
Invalid IDs: []

Q5: What is a Risk-o-meter?
Valid IDs: 3
Invalid IDs: []

Q6: What is the difference between direct and regular mutual fund plans?
Valid IDs: 2
Invalid IDs: []

Q7: What is NAV?
Valid IDs: 1
Invalid IDs: []

Q8: What is the difference between primary and secondary markets?
Valid IDs: 2
Invalid IDs: []

Q9: What is an IPO, and how does it work?
Valid IDs: 3
Invalid IDs: []

Q10: What is a demat account, and why is it required?
Valid IDs: 3
Invalid IDs: []


In [11]:
nism_questions = {
    "Q11": {
        "question": "What are derivatives, and what is their purpose?",
        "keywords": ["derivatives", "underlying asset", "financial contracts"]
    },
    "Q12": {
        "question": "What is a futures contract?",
        "keywords": ["futures contract", "obligation", "buy", "sell"]
    },
    "Q13": {
        "question": "What is an options contract?",
        "keywords": ["options contract", "right", "obligation", "call option", "put option"]
    },
    "Q14": {
        "question": "What is the difference between call options and put options?",
        "keywords": ["call option", "put option", "buying", "selling"]
    },
    "Q15": {
        "question": "What is the difference between futures and options contracts?",
        "keywords": ["futures", "options", "obligation", "right"]
    },
    "Q16": {
        "question": "What is a lot size in derivatives trading?",
        "keywords": ["lot size", "market lot", "contract size"]
    },
    "Q17": {
        "question": "What is margin in futures trading?",
        "keywords": ["initial margin", "margin", "futures"]
    },
    "Q18": {
        "question": "What is mark-to-market settlement?",
        "keywords": ["mark to market", "daily settlement", "MTM"]
    },
    "Q19": {
        "question": "What is an option's strike price and expiry date?",
        "keywords": ["strike price", "expiry date", "expiration"]
    },
    "Q20": {
        "question": "What are the different types of risks involved in derivatives trading?",
        "keywords": ["market risk", "liquidity risk", "credit risk", "operational risk"]
    }
}

print("Total NISM evaluation questions:", len(nism_questions))

Total NISM evaluation questions: 10


In [12]:
print("\n========== NISM GROUND-TRUTH SEARCH ==========")

for q_id, details in nism_questions.items():

    print(f"\n{'=' * 70}")
    print(f"{q_id}: {details['question']}")
    print("=" * 70)

    matches = []

    for chunk in chunks:

        # Search only the NISM book
        if "NISM-Series-VIII" not in chunk["document_name"]:
            continue

        text = chunk["text"].lower()

        if any(
            keyword.lower() in text
            for keyword in details["keywords"]
        ):
            matches.append(chunk)

    print(f"Matching chunks: {len(matches)}")

    for chunk in matches[:10]:
        print(f"\nChunk ID: {chunk['chunk_id']}")
        print(f"Document: {chunk['document_name']}")
        print(f"Page: {chunk['page_number']}")
        print(f"Text: {chunk['text'][:700]}")


========== NISM GROUND-TRUTH SEARCH ==========

Q11: What are derivatives, and what is their purpose?
Matching chunks: 252

Chunk ID: bdd630c10cdeca875d48fd9119991337
Document: 12 NISM-Series-VIII Equity Derivatives Certification Examination.pdf
Page: 2
Text: 1 
 
Workbook for 
NISM-Series-VIII: 
Equity Derivatives 
Certification Examination 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
National Institute of Securities Markets 
www.nism.ac.in

Chunk ID: 0b04665af30f595348f4c672e3a4ec14
Document: 12 NISM-Series-VIII Equity Derivatives Certification Examination.pdf
Page: 3
Text: 2 
 
This workbook has been developed to assist candidates i n preparing for the National 
Institute of Securities Markets (NISM) NISM -Series-VIII: Equity Derivatives Certification 
Examination (NISM-Series-VIII: ED Examination). 
 
 
Workbook Version: March 20261 
 
 
 
 
Published by: 
National Institute of Securities Markets 
© National Institute of Securities Markets, 2024 
5th Floor, NCL Co-operat

In [13]:
q11_ids = [
    "d8a0eadb815df00bf7108cfa87a2a9ff",
    "b1a3f1378c3bf2ad58bafc7f03e3d9ed",
    "0b04665af30f595348f4c672e3a4ec14"
]

for chunk in chunks:
    if chunk["chunk_id"] in q11_ids:
        print("\nChunk ID:", chunk["chunk_id"])
        print("Page:", chunk["page_number"])
        print("Text:\n", chunk["text"])
        print("-" * 80)


Chunk ID: 0b04665af30f595348f4c672e3a4ec14
Page: 3
Text:
 2 
 
This workbook has been developed to assist candidates i n preparing for the National 
Institute of Securities Markets (NISM) NISM -Series-VIII: Equity Derivatives Certification 
Examination (NISM-Series-VIII: ED Examination). 
 
 
Workbook Version: March 20261 
 
 
 
 
Published by: 
National Institute of Securities Markets 
© National Institute of Securities Markets, 2024 
5th Floor, NCL Co-operative Society, 
Plot No. C-6 E-Block, Bandra Kurla Complex,
--------------------------------------------------------------------------------

Chunk ID: d8a0eadb815df00bf7108cfa87a2a9ff
Page: 17
Text:
 16 
 
of price move), liquidity risk (inability to exit from a position), legal or regulatory risk 
(enforceability of contracts), operational risk (fraud, inadequate documentation, 
improper execution, etc.) and may not be an appropriate avenue for someone of limited 
resources, or having limited trading experience , or with low risk

In [14]:
for chunk in chunks:

    if (
        "NISM-Series-VIII" in chunk["document_name"]
        and 10 <= chunk["page_number"] <= 14
    ):
        print("\nChunk ID:", chunk["chunk_id"])
        print("Page:", chunk["page_number"])
        print("Text:\n", chunk["text"])
        print("-" * 80)


Chunk ID: 4a76817ad398df3e1a5656767764ca2e
Page: 10
Text:
 9 
 
7.9 Settlement of running account of Client’s funds lying with the TM: ....................... 170 
7.10 Settlement Guarantee Fund and Investor Protection Fund: .................................. 170 
7.11 Cyber Security & Cyber Resilience framework (CSCRF) for Stock Brokers / 
Depository Participants ................................................................................................... 172
--------------------------------------------------------------------------------

Chunk ID: 50b254bced10e162cb14c676a8c4e682
Page: 10
Text:
 CHAPTER 8: LEGAL AND REGULATORY ENVIRONMENT ............................................ 174 
8.1 Securities Contracts (Regulation) Act, 1956 ............................................................. 174 
8.2 Securities and Exchange Board of India Act, 1992 ................................................... 175 
8.3 Regulations in Trading .........................................

In [15]:
q12_ids = [
    "226c2532dd18b0582c3924fa99c101a5",
    "fa0f9831c4670dcff3bfaac8da0da5e1"
]

for chunk in chunks:

    if chunk["chunk_id"] in q12_ids:
        print("\nChunk ID:", chunk["chunk_id"])
        print("Page:", chunk["page_number"])
        print("Text:\n", chunk["text"])
        print("-" * 80)


Chunk ID: 226c2532dd18b0582c3924fa99c101a5
Page: 14
Text:
 negotiated between two parties, the terms and conditions of contracts are customized. 
These are Over-the-counter (OTC) contracts. 
Futures 
A futures co ntract is similar to a forward, except that the deal is made through an 
organized and regulated exchange rather than being negotiated directly between two 
parties. Futures are also standardized contracts (in terms of their lot size, maturity date, 
etc.) so that they can be traded on the exchange. Indeed, we may say futures are
--------------------------------------------------------------------------------

Chunk ID: fa0f9831c4670dcff3bfaac8da0da5e1
Page: 32
Text:
 price is the value of the Nifty index on October 03, 2025 which is 24894.25. 
Contract multiplier or Contract Size: Futures contracts are traded i n lots. The lot size or 
contract size for the index and stock futures is determined by the exchange. Contract sizes 
are different for each stock and index traded in

In [16]:
q13_ids = [
    "b67ce6818abb002c39cff9667c2eeb48"
]

for chunk in chunks:

    if chunk["chunk_id"] in q13_ids:
        print("\nChunk ID:", chunk["chunk_id"])
        print("Page:", chunk["page_number"])
        print("Text:\n", chunk["text"])


Chunk ID: b67ce6818abb002c39cff9667c2eeb48
Page: 9
Text:
 8 
 
4.4 Intrinsic value and time value of an option ................................................................ 64 
4.5 Payoff Charts for Options ............................................................................................ 65 
4.6 Distinction between futures and options contracts ................................................... 72 
4.7 Basics of Option Pricing and Option Greeks ............................................................... 74


In [17]:
keywords = ["call option", "put option"]

for chunk in chunks:
    if "NISM-Series-VIII" not in chunk["document_name"]:
        continue

    if 58 <= chunk["page_number"] <= 64:
        text = chunk["text"].lower()

        if all(keyword in text for keyword in keywords):
            print("\nChunk ID:", chunk["chunk_id"])
            print("Page:", chunk["page_number"])
            print("Text:\n", chunk["text"])
            print("-" * 80)


Chunk ID: a2263965cc287fdba329bcb5abe188ae
Page: 59
Text:
 The option buyer or holder has the right but no obligation with regards to buying or selling 
the underlying asset, while the option writer has the obligation in the contract. Therefore, 
the option buyer will exercise his option only when the situation is favourable to him. On 
the other hand, the option writer is legally bound to honour the contract whenever the 
option buyer decides to exercise his option. 
Options may be categorized into two main types: 
• Call Options 
• Put Options
--------------------------------------------------------------------------------

Chunk ID: 7c7ddc263af5b830b0bab32156a4f406
Page: 59
Text:
 Options may be categorized into two main types: 
• Call Options 
• Put Options 
An option, which gives the buyer/holder a right to buy the underlying asset, is called a call 
option; and an option which gives the buyer/holder a right to sell the underlying asset, is 
called a ’put option’. 
 
LEARNING OBJ

In [18]:
keywords = ["futures", "options"]

for chunk in chunks:
    if "NISM-Series-VIII" not in chunk["document_name"]:
        continue

    if 12 <= chunk["page_number"] <= 25:
        text = chunk["text"].lower()

        if all(keyword in text for keyword in keywords):
            print("\nChunk ID:", chunk["chunk_id"])
            print("Page:", chunk["page_number"])
            print("Text:\n", chunk["text"])
            print("-" * 80)


Chunk ID: 61dbe7b50873cfef68f89034afbc5609
Page: 13
Text:
 12 
 
In 1919, Chicago Butter and Egg Board, a spin -off of CBOT, was reorganised to allow 
futures trading. Later its name was changed to Chicago Mercantile Exchange (CME). 
In 1972, Chicago Mercantile Exchange introduced International Monetary Market (IMM), 
which allowed trading in currency futures. 
In 1973, Chicago Board Options Exchange (CBOE) became the first marketplace for trading 
listed options.
--------------------------------------------------------------------------------

Chunk ID: 23bd8731b06374c7a1077efe2426e298
Page: 13
Text:
 listed options. 
In 1975, CBOT introduced Treasury bill futures contract. It was the first successful pure 
interest rate futures. 
In 1977, CBOT introduced T-bond futures contract. 
In 1982, CME introduced Eurodollar futures contract. 
In 1982, Kansas City Board of Trade launched the first stock index futures. 
In 1983, Chicago Board Options Exchange (CBOE) introduced option on stock i

In [19]:
for chunk in chunks:
    if "NISM-Series-VIII" not in chunk["document_name"]:
        continue

    if 14 <= chunk["page_number"] <= 16:
        print("\nChunk ID:", chunk["chunk_id"])
        print("Page:", chunk["page_number"])
        print("Text:\n", chunk["text"])
        print("-" * 80)


Chunk ID: a78bd12480517c3b9cbe89d60b2c1705
Page: 14
Text:
 13 
 
for charging initial margins, membership details a nd net-worth criterion, deposit 
requirements and real time monitoring of positions requirements. 
In 1999, The Securities Contract Regulation Act (SCRA) was amended to include 
“derivatives” within the domain of ‘securities’ and a regulatory framework was developed 
for governing derivatives trading. In March 2000, the government repealed a three -
decade-old notification, which prohibited forward trading in securities.
--------------------------------------------------------------------------------

Chunk ID: 0134bbb2967675fceb2a84bac6fa2abc
Page: 14
Text:
 decade-old notification, which prohibited forward trading in securities. 
The exchange traded derivatives started in India in June 2000 with SEBI permitting BSE 
and NSE to introduce the equity derivatives segment. To begin with, SEBI approved 
trading in index futures contracts based on Nifty and Sensex, which comm

In [20]:
for chunk in chunks:
    if "NISM-Series-VIII" not in chunk["document_name"]:
        continue

    text = chunk["text"].lower()

    if any(keyword in text for keyword in ["lot size", "market lot", "contract size"]):
        print("\nChunk ID:", chunk["chunk_id"])
        print("Page:", chunk["page_number"])
        print("Text:\n", chunk["text"])
        print("-" * 80)


Chunk ID: 226c2532dd18b0582c3924fa99c101a5
Page: 14
Text:
 negotiated between two parties, the terms and conditions of contracts are customized. 
These are Over-the-counter (OTC) contracts. 
Futures 
A futures co ntract is similar to a forward, except that the deal is made through an 
organized and regulated exchange rather than being negotiated directly between two 
parties. Futures are also standardized contracts (in terms of their lot size, maturity date, 
etc.) so that they can be traded on the exchange. Indeed, we may say futures are
--------------------------------------------------------------------------------

Chunk ID: a52940cc7b88a835d26da6e245b31e3b
Page: 31
Text:
 The exchange decides all terms and conditions of futures contracts other than the price 
of the futures contract. These terms and conditions are known as ‘contract specifications. 
Contract specifications include the salient features of a derivative s contract like contract 
maturity, contract multiplier or cont

In [21]:
for chunk in chunks:
    if "NISM-Series-VIII" not in chunk["document_name"]:
        continue

    text = chunk["text"].lower()

    if any(keyword in text for keyword in [
        "initial margin",
        "margin account",
        "margin requirement",
        "futures contract"
    ]):
        print("\nChunk ID:", chunk["chunk_id"])
        print("Page:", chunk["page_number"])
        print("Text:\n", chunk["text"])
        print("-" * 80)


Chunk ID: 81d7c5dd2a69adb43f687363006b6cce
Page: 8
Text:
 2.7 Application of Indices .................................................................................................. 25 
CHAPTER 3: INTRODUCTION TO FORWARDS AND FUTURES ....................................... 28 
3.1 Introduction to forward contracts .............................................................................. 28 
3.2 Futures contracts ........................................................................................................ 29
--------------------------------------------------------------------------------

Chunk ID: d178ba897c2d89254f5e55e84009fd5d
Page: 8
Text:
 3.3 Contract specifications of futures contracts ............................................................... 30 
3.4 Some important terminology associated with futures contracts ............................... 34 
3.5 Differences between Forwards and Futures .............................................................. 38

In [22]:
for chunk in chunks:
    if "NISM-Series-VIII" not in chunk["document_name"]:
        continue

    text = chunk["text"].lower()

    if any(keyword in text for keyword in [
        "marking to market",
        "mark to market",
        "mtm settlement",
        "daily settlement"
    ]):
        print("\nChunk ID:", chunk["chunk_id"])
        print("Page:", chunk["page_number"])
        print("Text:\n", chunk["text"])
        print("-" * 80)


Chunk ID: 893aba4c0d6c9a016fe97edd83104e84
Page: 9
Text:
 7.4 Settlement Mechanism ............................................................................................. 153 
7.5 Risk Management ..................................................................................................... 158 
7.6 Margining and mark to market under SPAN............................................................. 159 
7.7 Position limits ............................................................................................................ 166
--------------------------------------------------------------------------------

Chunk ID: b957cf668e1d35cc886a8b0ff0c878bd
Page: 31
Text:
 • Quantity decided today (standardized) 
Limitations of Futures Contract 
• As futures are standardized contracts introduced by the exchanges, they too have 
certain limitations in the context of limited maturities, limited underlying set, lack 
of flexibility in contract design and increased administrati

In [23]:
for chunk in chunks:
    if "NISM-Series-VIII" not in chunk["document_name"]:
        continue

    text = chunk["text"].lower()

    if any(keyword in text for keyword in [
        "strike price",
        "expiry date",
        "expiration date",
        "expiration"
    ]):
        print("\nChunk ID:", chunk["chunk_id"])
        print("Page:", chunk["page_number"])
        print("Text:\n", chunk["text"])
        print("-" * 80)


Chunk ID: c405e8fa9c3bf4e714e6b353afbf13f9
Page: 31
Text:
 contract specifications with an example of a Nifty futures contract traded on the NSE. 
Quotes given on the NSE website for Nifty futures as on Oct 03, 2025 
1. Instrument type : Index futures 
2. Underlying asset : Nifty 50 
3. Expiry date : Oct 28, 2025 
4. Open price (in Rs.) : 24950.40 
5. High price (in Rs.) : 25020 
6. Low price (in Rs.) : 24878.60 
7. Closing price (in Rs.) : 25006.60 
8. Underlying value (in Rs.) : 24894.25
--------------------------------------------------------------------------------

Chunk ID: 122ae120a4269cf99c64b51e34fba8d7
Page: 32
Text:
 trading on monthly as well as weekly futures contracts. 
Expiration Day: This is the day on which a derivative contract ceases to exist. It is the last 
trading day of the contract. On expiry date, all the contracts are compulsorily settled. If a 
position is to be continued, then it must be rolled over to another futures contract of the 
same underlying. For a

In [24]:
for chunk in chunks:
    if "NISM-Series-VIII" not in chunk["document_name"]:
        continue

    text = chunk["text"].lower()

    if any(keyword in text for keyword in [
        "market risk",
        "liquidity risk",
        "credit risk",
        "operational risk",
        "counterparty risk"
    ]):
        print("\nChunk ID:", chunk["chunk_id"])
        print("Page:", chunk["page_number"])
        print("Text:\n", chunk["text"])
        print("-" * 80)


Chunk ID: 23d3432a11a13ca208e6407c6c4ef16f
Page: 16
Text:
 • It enables the shift of speculative trades from the unorganized market to the 
organized market. Risk management mechanism and surveillance of activities of 
various participants in the organized space provide stability to the financial system. 
1.7 Various risks faced by the participants in derivatives 
Market participants must understand that derivatives, being leveraged instruments, have 
risks like counterparty risk (default by counterparty), price risk (loss on position because
--------------------------------------------------------------------------------

Chunk ID: d8a0eadb815df00bf7108cfa87a2a9ff
Page: 17
Text:
 16 
 
of price move), liquidity risk (inability to exit from a position), legal or regulatory risk 
(enforceability of contracts), operational risk (fraud, inadequate documentation, 
improper execution, etc.) and may not be an appropriate avenue for someone of limited 
resources, or having limited trading ex

In [25]:
ground_truth.update({

    "Q11": {
        "question": "What are derivatives, and what is their purpose?",
        "relevant_chunk_ids": [
            "1b2789cfc34fa2e8854c0a423ac631d7",
            "b1a3f1378c3bf2ad58bafc7f03e3d9ed"
        ]
    },

    "Q12": {
        "question": "What is a futures contract?",
        "relevant_chunk_ids": [
            "226c2532dd18b0582c3924fa99c101a5",
            "fa0f9831c4670dcff3bfaac8da0da5e1"
        ]
    },

    "Q13": {
        "question": "What is an options contract?",
        "relevant_chunk_ids": [
            "ba1d25f9b29ff0ffc873132e3ad49127"
        ]
    },

    "Q14": {
        "question": "What is the difference between call options and put options?",
        "relevant_chunk_ids": [
            "7c7ddc263af5b830b0bab32156a4f406"
        ]
    },

    "Q15": {
        "question": "What is the difference between futures and options contracts?",
        "relevant_chunk_ids": [
            "226c2532dd18b0582c3924fa99c101a5",
            "ba1d25f9b29ff0ffc873132e3ad49127"
        ]
    },

    "Q16": {
        "question": "What is a lot size in derivatives trading?",
        "relevant_chunk_ids": [
            "fa0f9831c4670dcff3bfaac8da0da5e1",
            "8046e5f5d528ae39e56ac962a34d774d"
        ]
    },

    "Q17": {
        "question": "What is margin in futures trading?",
        "relevant_chunk_ids": [
            "c6fc354bd8c815bcc619fe96afd7735c",
            "3ea00053dfc1a8cafba293ba82cdd9fd",
            "7bb72571c009d141badbf7144e40b250"
        ]
    },

    "Q18": {
        "question": "What is mark-to-market settlement?",
        "relevant_chunk_ids": [
            "3ad1fdc302372880f33ca6c0c8a7bf3d",
            "ff457ecea01bd961ba7887f7530808be",
            "f7ed77a56b1bf745d38fbe519dd5663d",
            "de2be167246d198fa22f95d7a318ef5c"
        ]
    },

    "Q19": {
        "question": "What is an option's strike price and expiry date?",
        "relevant_chunk_ids": [
            "6c06f0e6e945ee4887cc3c53c92c1ab9",
            "e4aa0966fce50b77b13983c254cfe22a"
        ]
    },

    "Q20": {
        "question": "What are the different types of risks involved in derivatives trading?",
        "relevant_chunk_ids": [
            "23d3432a11a13ca208e6407c6c4ef16f",
            "d8a0eadb815df00bf7108cfa87a2a9ff",
            "8dc887027f3e3afc804436da7784d365",
            "1241139e45ca292edadb39a66c0e19fc",
            "9d6d43a45f065529c7c8976bc253447e"
        ]
    }
})

print("Total questions:", len(ground_truth))

Total questions: 20


In [26]:
existing_chunk_ids = {
    chunk["chunk_id"] for chunk in chunks
}

total_valid = 0
total_invalid = 0

for q_id, details in ground_truth.items():

    valid_ids = []
    invalid_ids = []

    for chunk_id in details["relevant_chunk_ids"]:
        if chunk_id in existing_chunk_ids:
            valid_ids.append(chunk_id)
        else:
            invalid_ids.append(chunk_id)

    total_valid += len(valid_ids)
    total_invalid += len(invalid_ids)

    print(f"\n{q_id}")
    print("Valid IDs:", len(valid_ids))
    print("Invalid IDs:", len(invalid_ids))

    if invalid_ids:
        print("Invalid chunk IDs:", invalid_ids)

print("\n" + "=" * 40)
print("GROUND-TRUTH VALIDATION SUMMARY")
print("=" * 40)
print("Total questions:", len(ground_truth))
print("Valid chunk IDs:", total_valid)
print("Invalid chunk IDs:", total_invalid)


Q1
Valid IDs: 1
Invalid IDs: 0

Q2
Valid IDs: 5
Invalid IDs: 0

Q3
Valid IDs: 4
Invalid IDs: 0

Q4
Valid IDs: 3
Invalid IDs: 0

Q5
Valid IDs: 3
Invalid IDs: 0

Q6
Valid IDs: 2
Invalid IDs: 0

Q7
Valid IDs: 1
Invalid IDs: 0

Q8
Valid IDs: 2
Invalid IDs: 0

Q9
Valid IDs: 3
Invalid IDs: 0

Q10
Valid IDs: 3
Invalid IDs: 0

Q11
Valid IDs: 2
Invalid IDs: 0

Q12
Valid IDs: 2
Invalid IDs: 0

Q13
Valid IDs: 1
Invalid IDs: 0

Q14
Valid IDs: 1
Invalid IDs: 0

Q15
Valid IDs: 2
Invalid IDs: 0

Q16
Valid IDs: 2
Invalid IDs: 0

Q17
Valid IDs: 3
Invalid IDs: 0

Q18
Valid IDs: 4
Invalid IDs: 0

Q19
Valid IDs: 2
Invalid IDs: 0

Q20
Valid IDs: 5
Invalid IDs: 0

GROUND-TRUTH VALIDATION SUMMARY
Total questions: 20
Valid chunk IDs: 51
Invalid chunk IDs: 0


In [27]:
import sys
sys.path.append("src")

from document_qa_chatbot.retrieval import retrieve_relevant_chunks
import pandas as pd

c:\Users\Nirmal\Desktop\document_qa_chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 338/338 [00:06<00:00, 54.38it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4393.21it/s]


In [28]:
import sys
sys.path.append("src")

from document_qa_chatbot.retrieval import (
    generate_embeddings,
    create_faiss_index,
    create_bm25_index
)

# Step 1: Generate embeddings
chunk_embeddings = generate_embeddings(chunks)

# Step 2: Create FAISS index
index = create_faiss_index(chunk_embeddings)

# Step 3: Create BM25 index
bm25_index = create_bm25_index(chunks)

print("All retrieval objects created successfully!")

All retrieval objects created successfully!


In [29]:
print("index:", index is not None)
print("chunk_embeddings:", chunk_embeddings is not None)
print("bm25_index:", bm25_index is not None)

index: True
chunk_embeddings: True
bm25_index: True


In [ ]:
#No Need to Rerun
K_VALUES = [1, 3, 5, 10]
MAX_K = max(K_VALUES)

results = []

for q_id, details in ground_truth.items():

    question = details["question"]
    relevant_ids = set(details["relevant_chunk_ids"])

    # Retrieve ranked chunks
    retrieved_chunks = retrieve_relevant_chunks(
        question=question,
        index=index,
        chunks=chunks,
        chunk_embeddings=chunk_embeddings,
        bm25_index=bm25_index,
        top_k=MAX_K,
        rerank_top_k=MAX_K,
        bm25_top_k=MAX_K,
        rrf_top_k=MAX_K
    )

    # Extract retrieved chunk IDs
    retrieved_ids = [
        chunk["chunk_id"] for chunk in retrieved_chunks
    ]

    # Calculate Recall@K
    row = {
        "Question ID": q_id,
        "Question": question
    }

    for k in K_VALUES:
        top_k_ids = set(retrieved_ids[:k])

        relevant_retrieved = len(
            relevant_ids.intersection(top_k_ids)
        )

        recall = relevant_retrieved / len(relevant_ids)

        row[f"Recall@{k}"] = recall

    results.append(row)

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))


--- QUERY REWRITING ---
Original Question: What is a mutual fund?
Rewritten Question: What is a mutual fund?

--- FAISS RETRIEVAL ---
Retrieved chunks: 10

--- BM25 RETRIEVAL ---
Retrieved chunks: 10

--- RRF FUSION ---
Rank: 1 | RRF Score: 0.032522 | Page: 97
Rank: 2 | RRF Score: 0.030331 | Page: 69
Rank: 3 | RRF Score: 0.016393 | Page: 87
Rank: 4 | RRF Score: 0.016129 | Page: 71
Rank: 5 | RRF Score: 0.015873 | Page: 219
Rank: 6 | RRF Score: 0.015873 | Page: 184
Rank: 7 | RRF Score: 0.015625 | Page: 69
Rank: 8 | RRF Score: 0.015385 | Page: 104
Rank: 9 | RRF Score: 0.015385 | Page: 67
Rank: 10 | RRF Score: 0.015152 | Page: 96

--- CROSSENCODER RERANKING ---

Rank: 1
FAISS Score: N/A
RRF Score: 0.032522
CrossEncoder Score: 7.8503
Document: Investor Awareness Booklet (English).pdf
Page: 97

Rank: 2
FAISS Score: 0.6867656707763672
RRF Score: 0.015873
CrossEncoder Score: 6.4757
Document: Investor Awareness Booklet (English).pdf
Page: 219

Rank: 3
FAISS Score: N/A
RRF Score: 0.030331
Cross

KeyboardInterrupt: 

In [31]:
print("index:", index is not None)
print("chunk_embeddings:", chunk_embeddings is not None)
print("bm25_index:", bm25_index is not None)

index: True
chunk_embeddings: True
bm25_index: True


In [32]:
# Get Q2 details
q2 = ground_truth["Q2"]

print("QUESTION:", q2["question"])

print("\nEXPECTED RELEVANT CHUNKS")
print("=" * 60)

for chunk_id in q2["relevant_chunk_ids"]:
    chunk = next(
        (c for c in chunks if c["chunk_id"] == chunk_id),
        None
    )

    if chunk:
        print("\nPage:", chunk["page_number"])
        print("Chunk ID:", chunk["chunk_id"])
        print("Text:", chunk["text"][:1000])


print("\nRETRIEVED TOP 10 CHUNKS")
print("=" * 60)

retrieved = retrieve_relevant_chunks(
    question=q2["question"],
    index=index,
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    bm25_index=bm25_index,
    top_k=10,
    rerank_top_k=10,
    bm25_top_k=10,
    rrf_top_k=10
)

for rank, chunk in enumerate(retrieved, start=1):
    print(f"\nRank: {rank}")
    print("Page:", chunk["page_number"])
    print("Chunk ID:", chunk["chunk_id"])
    print("Text:", chunk["text"][:1000])

QUESTION: How do I choose a mutual fund?

EXPECTED RELEVANT CHUNKS

Page: 104
Chunk ID: 54fcbd387ffb7b8256e048b0a7758077
Text: fund for a fifteen -year goal” and then picks a reasonable diversified fu nd will do far better 
than someone who picks last year’s top performer without knowing why they are buying 
equity at all. 
The categories 
SEBI has standardised mutual fund categories so that schemes with the same label do broadly 
the same thing across fund houses. Here is the map. 
By structure 
Open-ended: You can buy or redeem units at any time, at the prevailing NAV. No fixed

Page: 111
Chunk ID: 24f67f008628cb5a7df574645e77938a
Text: for three years, a hybrid or short duration debt fund. 
4. Shortlist three or four schemes in that category from established fund houses. 
5. Compare: rolling returns against the benchmark over five years or more, TER, portfolio 
concentration, fund manager tenure, and consistency during past falls. 
6. Read the SID — objective, asset allocation, risk

In [33]:
q2 = ground_truth["Q2"]

for chunk_id in q2["relevant_chunk_ids"]:

    chunk = next(
        c for c in chunks
        if c["chunk_id"] == chunk_id
    )

    print("\n" + "=" * 80)
    print("PAGE:", chunk["page_number"])
    print("CHUNK ID:", chunk["chunk_id"])
    print("=" * 80)
    print(chunk["text"])


PAGE: 104
CHUNK ID: 54fcbd387ffb7b8256e048b0a7758077
fund for a fifteen -year goal” and then picks a reasonable diversified fu nd will do far better 
than someone who picks last year’s top performer without knowing why they are buying 
equity at all. 
The categories 
SEBI has standardised mutual fund categories so that schemes with the same label do broadly 
the same thing across fund houses. Here is the map. 
By structure 
Open-ended: You can buy or redeem units at any time, at the prevailing NAV. No fixed

PAGE: 111
CHUNK ID: 24f67f008628cb5a7df574645e77938a
for three years, a hybrid or short duration debt fund. 
4. Shortlist three or four schemes in that category from established fund houses. 
5. Compare: rolling returns against the benchmark over five years or more, TER, portfolio 
concentration, fund manager tenure, and consistency during past falls. 
6. Read the SID — objective, asset allocation, risk factors. 
7. Check the Risk-o-meter against your risk tolerance. 
8. Invest, a

In [34]:
from document_qa_chatbot.retrieval import rewrite_query

question = ground_truth["Q2"]["question"]
rewritten_question = rewrite_query(question)

print("Original question:", question)
print("Rewritten question:", rewritten_question)

print("\nEXPECTED CHUNK IDs:")
for chunk_id in ground_truth["Q2"]["relevant_chunk_ids"]:
    print(chunk_id)

Original question: How do I choose a mutual fund?
Rewritten question: What factors should I consider when selecting a mutual fund?

EXPECTED CHUNK IDs:
54fcbd387ffb7b8256e048b0a7758077
24f67f008628cb5a7df574645e77938a
46286916f98b3212e7d506ae7dc528e1
a9d76003d839997fa29fad3fa59d29d5
7d6bb1dfcecc63108956b65382cb412e


In [35]:
import inspect
import document_qa_chatbot.retrieval as retrieval

for name, obj in inspect.getmembers(retrieval, inspect.isfunction):
    print(name)

combine_results_rrf
create_bm25_index
create_faiss_index
generate_embeddings
generate_question_embedding
get_user_question
rerank_chunks
retrieve_bm25_chunks
retrieve_relevant_chunks
rewrite_query


In [36]:
import inspect
import document_qa_chatbot.retrieval as retrieval

print("BM25 function:")
print(inspect.signature(retrieval.retrieve_bm25_chunks))

print("\nQuestion embedding function:")
print(inspect.signature(retrieval.generate_question_embedding))

print("\nHybrid retrieval function:")
print(inspect.signature(retrieval.retrieve_relevant_chunks))

BM25 function:
(question, chunks, bm25_index, top_k=5)

Question embedding function:
(question)

Hybrid retrieval function:
(question, index, chunks, chunk_embeddings, bm25_index, top_k=5, min_score=0.35, document_filter=None, rerank_top_k=3, bm25_top_k=5, rrf_top_k=5)


In [37]:
from document_qa_chatbot.retrieval import (
    generate_question_embedding,
    retrieve_bm25_chunks
)

In [38]:
# 1. FAISS retrieval
question_embedding = generate_question_embedding(rewritten_question)

faiss_scores, faiss_indices = index.search(
    question_embedding,
    10
)

print("FAISS RESULTS")
for rank, (score, idx) in enumerate(
    zip(faiss_scores[0], faiss_indices[0]), start=1
):
    if idx == -1:
        continue

    chunk = chunks[idx]

    print(
        rank,
        "Chunk ID:", chunk["chunk_id"],
        "Page:", chunk["page_number"],
        "Score:", round(float(score), 4)
    )


# 2. BM25 retrieval
bm25_results = retrieve_bm25_chunks(
    rewritten_question,
    chunks,
    bm25_index,
    top_k=10
)

print("\nBM25 RESULTS")
for rank, item in enumerate(bm25_results, start=1):
    print(
        rank,
        "Chunk ID:", item["chunk_id"],
        "Page:", item["page_number"]
    )

FAISS RESULTS
1 Chunk ID: d16610361810cd4a5274880793297905 Page: 104 Score: 0.6777
2 Chunk ID: f2b276f9d32082802223f3d4eaf28234 Page: 97 Score: 0.6666
3 Chunk ID: 5c573073f1fe2f7d5f4aed7e9b21aa06 Page: 104 Score: 0.6245
4 Chunk ID: 479b81c80a73449e0543d6caee6e9cc4 Page: 97 Score: 0.5846
5 Chunk ID: 8478fb5b1a0901c8bd8d3bae1147ea34 Page: 151 Score: 0.5702
6 Chunk ID: 2496292d664048e99d2a3cdb6c6b64c8 Page: 104 Score: 0.5665
7 Chunk ID: 93a8ddd3c068762ca2d0ac9484dc25bd Page: 69 Score: 0.5632
8 Chunk ID: 24f67f008628cb5a7df574645e77938a Page: 111 Score: 0.5619
9 Chunk ID: a05d56c555924ae6613107bb893cad94 Page: 96 Score: 0.5445
10 Chunk ID: 0076a087056d92b3adf4f28947bf7902 Page: 69 Score: 0.54

BM25 RESULTS
1 Chunk ID: 05d9df8d88ba051fcd5cc49378619bf5 Page: 151
2 Chunk ID: 49a50072b47d1643d086907b669b3372 Page: 198
3 Chunk ID: 617070eba98a6bd8d2a5b742430f82f2 Page: 198
4 Chunk ID: c72e14c296d2eb5973a59564ce3cae1b Page: 160
5 Chunk ID: 99ba137aec13d60f10419f6243de3992 Page: 154
6 Chunk ID: d

In [39]:
from rank_bm25 import BM25Okapi

question_tokens = rewritten_question.lower().split()

print("BM25 query tokens:")
print(question_tokens)

print("\nExpected chunk texts and their BM25 scores:")

for chunk_id in ground_truth["Q2"]["relevant_chunk_ids"]:

    chunk_index = next(
        i for i, c in enumerate(chunks)
        if c["chunk_id"] == chunk_id
    )

    score = bm25_index.get_scores(question_tokens)[chunk_index]

    print(
        "\nPage:", chunks[chunk_index]["page_number"],
        "\nBM25 Score:", score,
        "\nText:", chunks[chunk_index]["text"][:300]
    )

BM25 query tokens:
['what', 'factors', 'should', 'i', 'consider', 'when', 'selecting', 'a', 'mutual', 'fund?']

Expected chunk texts and their BM25 scores:

Page: 104 
BM25 Score: 4.7828600478766905 
Text: fund for a fifteen -year goal” and then picks a reasonable diversified fu nd will do far better 
than someone who picks last year’s top performer without knowing why they are buying 
equity at all. 
The categories 
SEBI has standardised mutual fund categories so that schemes with the same label do b

Page: 111 
BM25 Score: 1.6014799614500668 
Text: for three years, a hybrid or short duration debt fund. 
4. Shortlist three or four schemes in that category from established fund houses. 
5. Compare: rolling returns against the benchmark over five years or more, TER, portfolio 
concentration, fund manager tenure, and consistency during past falls.

Page: 106 
BM25 Score: 4.75279700118278 
Text: Every scheme publishes a monthly factsheet — a two - or three -page summary that fund 
houses 

In [40]:
bm25_scores = bm25_index.get_scores(question_tokens)

top_indices = sorted(
    range(len(bm25_scores)),
    key=lambda i: bm25_scores[i],
    reverse=True
)[:10]

print("BM25 TOP 10 WITH SCORES")

for rank, idx in enumerate(top_indices, start=1):
    chunk = chunks[idx]

    print(
        f"\nRank: {rank}",
        f"\nPage: {chunk['page_number']}",
        f"\nChunk ID: {chunk['chunk_id']}",
        f"\nScore: {bm25_scores[idx]:.4f}",
        f"\nText: {chunk['text'][:250]}"
    )

BM25 TOP 10 WITH SCORES

Rank: 1 
Page: 151 
Chunk ID: 05d9df8d88ba051fcd5cc49378619bf5 
Score: 14.3795 
Text: On major life 
events 
Marriage, child, job change, inheritance, serious illness — reassess risk 
capacity 
 
What a review is, and is not 
A review asks: Am I still on track for this goal? Has my allocation drifted from target? Has 
anything fundame

Rank: 2 
Page: 198 
Chunk ID: 49a50072b47d1643d086907b669b3372 
Score: 12.6297 
Text: • ☐ I have checked the Risk-o-meter against my risk tolerance 
• ☐ I can explain in two sentences what I am buying and why 
• ☐ I am not acting on a tip, and I am not in a hurry

Rank: 3 
Page: 198 
Chunk ID: 617070eba98a6bd8d2a5b742430f82f2 
Score: 12.4284 
Text: • ☐ I have registered a nominee 
• ☐ Two-factor authentication is enabled 
• ☐ SMS and email alerts are switched on 
• ☐ My mobile number and email are current on every account 
Before every investment 
• ☐ I know which goal this is for and its horiz

Rank: 4 
Page: 160 
Chunk ID: c72e

In [41]:
test_query = """
How to select a mutual fund scheme.
Compare investment objectives, fund categories,
investment horizon, risk tolerance, rolling returns,
expense ratio, fund manager, portfolio and Riskometer.
"""

test_tokens = test_query.lower().split()

test_scores = bm25_index.get_scores(test_tokens)

top_indices = sorted(
    range(len(test_scores)),
    key=lambda i: test_scores[i],
    reverse=True
)[:10]

for rank, idx in enumerate(top_indices, start=1):
    chunk = chunks[idx]

    print(
        f"\nRank: {rank}",
        f"\nPage: {chunk['page_number']}",
        f"\nChunk ID: {chunk['chunk_id']}",
        f"\nScore: {test_scores[idx]:.4f}"
    )


Rank: 1 
Page: 219 
Chunk ID: 881d3f1a57e70257298f09cdb2f2451d 
Score: 33.2870

Rank: 2 
Page: 220 
Chunk ID: 514e9d87142ff5e083047662ef39fce1 
Score: 32.6230

Rank: 3 
Page: 97 
Chunk ID: f2b276f9d32082802223f3d4eaf28234 
Score: 31.0862

Rank: 4 
Page: 120 
Chunk ID: 3731a52e3f565115d978d0cbf9bb447b 
Score: 29.0210

Rank: 5 
Page: 220 
Chunk ID: 042058c7db4879c0703a915729906b35 
Score: 28.6515

Rank: 6 
Page: 106 
Chunk ID: 46286916f98b3212e7d506ae7dc528e1 
Score: 28.5365

Rank: 7 
Page: 107 
Chunk ID: a9d76003d839997fa29fad3fa59d29d5 
Score: 26.8354

Rank: 8 
Page: 100 
Chunk ID: 2e5d4e65021661d42c0580c03acb9675 
Score: 25.0712

Rank: 9 
Page: 220 
Chunk ID: 087cd1c7124e408d5826c124d99f08bc 
Score: 24.8730

Rank: 10 
Page: 221 
Chunk ID: 3043f2f06723fed2a02d361e6c975550 
Score: 24.6638


In [42]:
test_embedding = generate_question_embedding(test_query)

test_faiss_scores, test_faiss_indices = index.search(
    test_embedding,
    10
)

print("FAISS RESULTS - EXPANDED QUERY")

for rank, (score, idx) in enumerate(
    zip(test_faiss_scores[0], test_faiss_indices[0]),
    start=1
):
    if idx == -1:
        continue

    chunk = chunks[idx]

    print(
        f"\nRank: {rank}",
        f"\nPage: {chunk['page_number']}",
        f"\nChunk ID: {chunk['chunk_id']}",
        f"\nScore: {score:.4f}"
    )

FAISS RESULTS - EXPANDED QUERY

Rank: 1 
Page: 104 
Chunk ID: d16610361810cd4a5274880793297905 
Score: 0.8201

Rank: 2 
Page: 104 
Chunk ID: 5c573073f1fe2f7d5f4aed7e9b21aa06 
Score: 0.7167

Rank: 3 
Page: 97 
Chunk ID: f2b276f9d32082802223f3d4eaf28234 
Score: 0.6819

Rank: 4 
Page: 111 
Chunk ID: 24f67f008628cb5a7df574645e77938a 
Score: 0.5791

Rank: 5 
Page: 104 
Chunk ID: 2496292d664048e99d2a3cdb6c6b64c8 
Score: 0.5788

Rank: 6 
Page: 110 
Chunk ID: a7950b85250217a36dc83800ee2e88ee 
Score: 0.5693

Rank: 7 
Page: 97 
Chunk ID: 479b81c80a73449e0543d6caee6e9cc4 
Score: 0.5566

Rank: 8 
Page: 111 
Chunk ID: 97ece0097e538aa9068d355b37c258cb 
Score: 0.5383

Rank: 9 
Page: 105 
Chunk ID: 12ec15569d753c5150e774dcda6e72c7 
Score: 0.5306

Rank: 10 
Page: 219 
Chunk ID: 881d3f1a57e70257298f09cdb2f2451d 
Score: 0.5303


In [43]:
import inspect

print(inspect.signature(retrieval.combine_results_rrf))

(faiss_results, bm25_results, k=60, top_k=5)


In [44]:
import inspect

print(inspect.getsource(retrieval.combine_results_rrf))

def combine_results_rrf(
    faiss_results,
    bm25_results,
    k=60,
    top_k=5
):
    rrf_scores = {}
    chunk_lookup = {}

    retrieval_results = [
        faiss_results,
        bm25_results
    ]

    for results in retrieval_results:

        for rank, chunk in enumerate(results, start=1):

            chunk_id = (
                chunk["document_name"],
                chunk["page_number"],
                chunk["text"]
            )

            score = 1 / (k + rank)

            rrf_scores[chunk_id] = (
                rrf_scores.get(chunk_id, 0) + score
            )

            chunk_lookup[chunk_id] = chunk

    ranked_ids = sorted(
        rrf_scores,
        key=lambda chunk_id: rrf_scores[chunk_id],
        reverse=True
    )

    combined_results = []

    for chunk_id in ranked_ids[:top_k]:

        result = chunk_lookup[chunk_id].copy()
        result["rrf_score"] = rrf_scores[chunk_id]

        combined_results.append(result)

    return combined_results



In [45]:
# Convert FAISS results into the format expected by RRF
faiss_results = []

for idx in test_faiss_indices[0]:
    if idx == -1:
        continue
    faiss_results.append(chunks[idx])


# Convert BM25 results into the same format
bm25_results = []

for idx in sorted(
    range(len(test_scores)),
    key=lambda i: test_scores[i],
    reverse=True
)[:10]:
    bm25_results.append(chunks[idx])


# Combine using RRF
rrf_results = retrieval.combine_results_rrf(
    faiss_results=faiss_results,
    bm25_results=bm25_results,
    k=60,
    top_k=10
)


# Display results
print("RRF RESULTS - EXPANDED QUERY")

for rank, chunk in enumerate(rrf_results, start=1):
    print(
        f"\nRank: {rank}",
        f"\nPage: {chunk['page_number']}",
        f"\nChunk ID: {chunk['chunk_id']}",
        f"\nRRF Score: {chunk['rrf_score']:.6f}"
    )

RRF RESULTS - EXPANDED QUERY

Rank: 1 
Page: 97 
Chunk ID: f2b276f9d32082802223f3d4eaf28234 
RRF Score: 0.031746

Rank: 2 
Page: 219 
Chunk ID: 881d3f1a57e70257298f09cdb2f2451d 
RRF Score: 0.030679

Rank: 3 
Page: 104 
Chunk ID: d16610361810cd4a5274880793297905 
RRF Score: 0.016393

Rank: 4 
Page: 104 
Chunk ID: 5c573073f1fe2f7d5f4aed7e9b21aa06 
RRF Score: 0.016129

Rank: 5 
Page: 220 
Chunk ID: 514e9d87142ff5e083047662ef39fce1 
RRF Score: 0.016129

Rank: 6 
Page: 111 
Chunk ID: 24f67f008628cb5a7df574645e77938a 
RRF Score: 0.015625

Rank: 7 
Page: 120 
Chunk ID: 3731a52e3f565115d978d0cbf9bb447b 
RRF Score: 0.015625

Rank: 8 
Page: 104 
Chunk ID: 2496292d664048e99d2a3cdb6c6b64c8 
RRF Score: 0.015385

Rank: 9 
Page: 220 
Chunk ID: 042058c7db4879c0703a915729906b35 
RRF Score: 0.015385

Rank: 10 
Page: 110 
Chunk ID: a7950b85250217a36dc83800ee2e88ee 
RRF Score: 0.015152


In [46]:
expected_answers = {
    "Q1": (
        "A mutual fund collects money from multiple investors and "
        "invests it as a single pool in a portfolio of securities. "
        "A professional fund manager manages the investments."
    ),

    "Q2": (
        "Choose a mutual fund by first identifying your financial goal "
        "and investment horizon. Select a suitable asset class and fund "
        "category. Then shortlist schemes and compare rolling returns, "
        "expense ratio, portfolio concentration, fund manager experience, "
        "investment objective and Risk-o-meter. Read the scheme documents "
        "and factsheet before investing."
    ),

    "Q3": (
        "A Systematic Investment Plan (SIP) invests a fixed amount at "
        "regular intervals, usually monthly, into a chosen mutual fund "
        "scheme. It automates investing and supports rupee cost averaging "
        "because more units are purchased when NAV is low and fewer when "
        "NAV is high."
    ),

    "Q4": (
        "Equity funds invest mainly in shares and aim for long-term "
        "capital appreciation. They have higher risk and higher long-term "
        "growth potential. Debt funds invest in fixed-income instruments "
        "such as government securities, treasury bills and corporate "
        "bonds. They generally have lower risk and aim for relatively "
        "stable returns, but still carry interest-rate and credit risks."
    ),

    "Q5": (
        "A Risk-o-meter is a standardised dial that displays a mutual "
        "fund scheme's risk level using a prescribed methodology. "
        "It has six levels, from Low to Very High, and is updated monthly. "
        "It helps investors understand scheme risk, but does not by "
        "itself determine whether a scheme suits their financial goals."
    ),

    "Q6": (
        "A Direct plan is purchased directly from the AMC and has no "
        "distributor commission, resulting in a lower expense ratio. "
        "A Regular plan is purchased through a distributor, bank, broker "
        "or advisor and includes distributor commission in its expense "
        "ratio. Both plans have identical portfolios, but Regular plans "
        "provide assistance and servicing."
    ),

    "Q7": (
        "Net Asset Value (NAV) is the per-unit value of a mutual fund "
        "scheme. It is calculated by subtracting liabilities from the "
        "total market value of the scheme's assets and dividing the "
        "result by the number of outstanding units."
    ),

    "Q8": (
        "The primary market is where securities are issued for the "
        "first time, allowing companies to raise money from investors. "
        "The secondary market is where existing securities are traded "
        "between investors."
    ),

    "Q9": (
        "An Initial Public Offering (IPO) is a public issue through "
        "which a company offers shares to investors. In a fresh issue, "
        "the company creates and sells new shares and receives the money "
        "for purposes such as expansion or debt repayment. Investors "
        "can apply through mechanisms such as ASBA and UPI."
    ),

    "Q10": (
        "A demat account holds securities in dematerialised, or "
        "electronic, form instead of physical paper certificates. "
        "It is used to hold and manage securities electronically. "
        "A Depository Participant, typically a bank or broker, opens "
        "and services the account."
    ),

    "Q11": (
        "A derivative is a financial contract or product whose value "
        "is derived from an underlying asset, such as shares, bonds, "
        "foreign exchange, metals or commodities. Derivatives are used "
        "by market participants for purposes including hedging, "
        "speculation and arbitrage."
    ),

    "Q12": (
        "A futures contract is an exchange-traded, standardised agreement "
        "similar to a forward contract. It is traded through an organised "
        "and regulated exchange and has standardised terms such as "
        "lot size and maturity date."
    ),

    "Q13": (
        "An options contract gives its buyer the right, but not the "
        "obligation, to buy or sell an underlying asset at a specified "
        "price on or before a stated date. The buyer pays a premium, "
        "while the option seller receives the premium and has an "
        "obligation if the buyer exercises the option."
    ),

    "Q14": (
        "A call option gives the buyer the right to buy the underlying "
        "asset. A put option gives the buyer the right to sell the "
        "underlying asset."
    ),

    "Q15": (
        "A futures contract creates an obligation for both parties "
        "to honour the contract. An options contract gives the buyer "
        "a right, but not an obligation, to buy or sell the underlying "
        "asset. The option seller has an obligation if the buyer "
        "exercises the option."
    ),

    "Q16": (
        "Lot size, also called contract size, is the number of units "
        "of the underlying asset represented by one derivatives contract. "
        "The exchange determines the lot size for futures contracts."
    ),

    "Q17": (
        "Initial margin is the amount deposited into a margin account "
        "when entering a futures contract. It protects the exchange "
        "against potential defaults. Both buyers and sellers of futures "
        "contracts must pay initial margin."
    ),

    "Q18": (
        "Mark-to-market (MTM) settlement is the process of adjusting "
        "profits and losses based on daily price changes in the "
        "underlying asset. The exchange collects MTM amounts from "
        "loss-making participants and pays the gains to profitable "
        "participants on a daily basis."
    ),

    "Q19": (
        "The strike price, also called the exercise price, is the price "
        "at which the underlying security may be purchased by a call "
        "option holder or sold by a put option holder. The expiry date "
        "is the date on which the derivatives contract expires."
    ),

    "Q20": (
        "The risks faced by derivatives market participants include "
        "counterparty risk, price risk, liquidity risk, legal or "
        "regulatory risk and operational risk. These include the risk "
        "of counterparty default, losses from price movements, inability "
        "to exit a position, contract enforceability issues, fraud "
        "and improper execution."
    )
}


# Add expected answers to the existing ground_truth dictionary
for q_id, answer in expected_answers.items():
    ground_truth[q_id]["expected_answer"] = answer


print("Expected answers added successfully!")
print("Total questions:", len(ground_truth))
print(
    "Questions with expected answers:",
    sum(
        "expected_answer" in details
        for details in ground_truth.values()
    )
)

Expected answers added successfully!
Total questions: 20
Questions with expected answers: 20


In [47]:
import inspect
import document_qa_chatbot.generation as generation

print("Available generation functions:")

for name, obj in inspect.getmembers(generation, inspect.isfunction):
    print(name, inspect.signature(obj))

Available generation functions:
create_prompt (question, retrieved_chunks)
generate_answer (question, retrieved_chunks)
rewrite_query (question)


In [48]:
import inspect

print(inspect.getsource(generation.generate_answer))

def generate_answer(question, retrieved_chunks):
    if not retrieved_chunks:
        return "No relevant information was found in the documents."

    messages = create_prompt(question, retrieved_chunks)

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[1]

    answer = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    )

    return answer.strip()



In [ ]:
#No rerun required
import pandas as pd

evaluation_results = []

for q_id, item in ground_truth.items():

    question = item["question"]

    # Retrieve relevant chunks using the RAG pipeline
    retrieved_chunks = retrieval.retrieve_relevant_chunks(
        question,
        index,
        chunks,
        chunk_embeddings,
        bm25_index,
        top_k=5,
        min_score=0.35,
        rerank_top_k=3,
        bm25_top_k=5,
        rrf_top_k=5
    )

    # Generate answer using the application function
    generated_answer = generation.generate_answer(
        question,
        retrieved_chunks
    )

    # Store evaluation details
    evaluation_results.append({
        "Question ID": q_id,
        "Question": question,
        "Expected Answer": item["expected_answer"],
        "Generated Answer": generated_answer,
        "Retrieved Chunks": len(retrieved_chunks),
        "Retrieved Context": "\n\n".join(
            chunk["text"] for chunk in retrieved_chunks
        )
    })

evaluation_df = pd.DataFrame(evaluation_results)

print("Answer generation completed!")
print("Total questions:", len(evaluation_df))

display(
    evaluation_df[
        ["Question ID", "Question", "Expected Answer", "Generated Answer"]
    ]
)


--- QUERY REWRITING ---
Original Question: What is a mutual fund?
Rewritten Question: What is a mutual fund?

--- FAISS RETRIEVAL ---
Retrieved chunks: 5

--- BM25 RETRIEVAL ---
Retrieved chunks: 5

--- RRF FUSION ---
Rank: 1 | RRF Score: 0.032522 | Page: 97
Rank: 2 | RRF Score: 0.016393 | Page: 87
Rank: 3 | RRF Score: 0.016129 | Page: 71
Rank: 4 | RRF Score: 0.015873 | Page: 219
Rank: 5 | RRF Score: 0.015873 | Page: 184

--- CROSSENCODER RERANKING ---

Rank: 1
FAISS Score: N/A
RRF Score: 0.032522
CrossEncoder Score: 7.8503
Document: Investor Awareness Booklet (English).pdf
Page: 97

Rank: 2
FAISS Score: 0.6867656707763672
RRF Score: 0.015873
CrossEncoder Score: 6.4757
Document: Investor Awareness Booklet (English).pdf
Page: 219

Rank: 3
FAISS Score: 0.705432653427124
RRF Score: 0.016129
CrossEncoder Score: 4.3884
Document: Investor Awareness Booklet (English).pdf
Page: 71


KeyboardInterrupt: 

In [50]:
print(len(evaluation_results) if "evaluation_results" in globals() else "evaluation_results not created")

0


In [54]:
completed_ids = {
    result["Question ID"]
    for result in evaluation_results
}

for q_id, item in ground_truth.items():

    # Stop when 20 unique questions are completed
    if len(completed_ids) >= 20:
        print("20 unique questions completed. Stopping.")
        break

    # Skip questions already evaluated
    if q_id in completed_ids:
        continue

    question = item["question"]

    print(f"Generating {q_id}: {question}")

    retrieved_chunks = retrieval.retrieve_relevant_chunks(
        question,
        index,
        chunks,
        chunk_embeddings,
        bm25_index,
        top_k=5,
        min_score=0.35,
        rerank_top_k=3,
        bm25_top_k=5,
        rrf_top_k=5
    )

    generated_answer = generation.generate_answer(
        question,
        retrieved_chunks
    )

    evaluation_results.append({
        "Question ID": q_id,
        "Question": question,
        "Expected Answer": item["expected_answer"],
        "Generated Answer": generated_answer,
        "Retrieved Chunks": len(retrieved_chunks),
        "Retrieved Context": "\n\n".join(
            chunk["text"] for chunk in retrieved_chunks
        )
    })

    completed_ids.add(q_id)

    print(f"Completed: {len(completed_ids)}/20")

# Final unique results
evaluation_df = (
    pd.DataFrame(evaluation_results)
    .drop_duplicates(subset=["Question ID"], keep="first")
)

print("\nFinal unique questions:", len(evaluation_df))

Generating Q4: What is the difference between equity funds and debt funds?

--- QUERY REWRITING ---
Original Question: What is the difference between equity funds and debt funds?
Rewritten Question: What are the differences between equity funds and debt funds?

--- FAISS RETRIEVAL ---
Retrieved chunks: 5

--- BM25 RETRIEVAL ---
Retrieved chunks: 5

--- RRF FUSION ---
Rank: 1 | RRF Score: 0.032522 | Page: 69
Rank: 2 | RRF Score: 0.016393 | Page: 69
Rank: 3 | RRF Score: 0.016129 | Page: 169
Rank: 4 | RRF Score: 0.015873 | Page: 105
Rank: 5 | RRF Score: 0.015873 | Page: 106

--- CROSSENCODER RERANKING ---

Rank: 1
FAISS Score: 0.7242680191993713
RRF Score: 0.016393
CrossEncoder Score: 7.8368
Document: Investor Awareness Booklet (English).pdf
Page: 69

Rank: 2
FAISS Score: N/A
RRF Score: 0.032522
CrossEncoder Score: 5.8806
Document: Investor Awareness Booklet (English).pdf
Page: 69

Rank: 3
FAISS Score: N/A
RRF Score: 0.016129
CrossEncoder Score: 3.2748
Document: Investor Awareness Booklet

In [55]:
print(len(evaluation_results) if "evaluation_results" in globals() else "evaluation_results not created")

20


In [56]:
evaluation_df = (
    pd.DataFrame(evaluation_results)
    .drop_duplicates(subset="Question ID", keep="first")
    .sort_values("Question ID")
    .reset_index(drop=True)
)

print("Total questions:", len(evaluation_df))
print("Unique question IDs:", evaluation_df["Question ID"].nunique())
print("Missing answers:", evaluation_df["Generated Answer"].isna().sum())

display(evaluation_df)

Total questions: 20
Unique question IDs: 20
Missing answers: 0


,Question ID,Question,Expected Answer,Generated Answer,Retrieved Chunks,Retrieved Context
0,Q1,What is a mutual fund?,A mutual fund collects money from multiple inv...,A mutual fund is a type of pooled investment v...,3,"• Choose between a Direct and a Regular plan, ..."
1,Q10,"What is a demat account, and why is it required?",A demat account holds securities in dematerial...,A demat account is an electronic record of sec...,3,your units are recorded in a Statement of Acco...
2,Q11,"What are derivatives, and what is their purpose?",A derivative is a financial contract or produc...,Derivatives are contracts or products whose va...,3,11 \n \nCHAPTER 1: BASICS OF DERIVATIVES \n \n...
3,Q12,What is a futures contract?,"A futures contract is an exchange-traded, stan...",A futures contract is an agreement made throug...,3,Futures contracts were created to overcome the...
4,Q13,What is an options contract?,"An options contract gives its buyer the right,...",An options contract is a financial instrument ...,3,options have emerged as a financial instrument...
5,Q14,What is the difference between call options an...,A call option gives the buyer the right to buy...,The difference between call options and put op...,3,Options may be categorized into two main types...
6,Q15,What is the difference between futures and opt...,A futures contract creates an obligation for b...,Futures contracts and options contracts differ...,3,"options expire with no value, the entire amoun..."
7,Q16,What is a lot size in derivatives trading?,"Lot size, also called contract size, is the nu...",A lot size in derivatives trading refers to th...,3,"introduction in the market. Further, the lot s..."
8,Q17,What is margin in futures trading?,Initial margin is the amount deposited into a ...,Margin in futures trading refers to the initia...,3,position in a single futures contract gives th...
9,Q18,What is mark-to-market settlement?,Mark-to-market (MTM) settlement is the process...,Mark-to-market (MTM) settlement is a process u...,3,Mark to Market (MTM) is a process by which mar...


In [57]:
print("Model:", type(generation.model).__name__)
print("Tokenizer:", type(generation.tokenizer).__name__)
print("Device:", generation.model.device)


Model: Qwen2ForCausalLM
Tokenizer: Qwen2Tokenizer
Device: cpu


In [59]:

import json
import re
import torch
import pandas as pd

# Store completed evaluations so the cell can resume if interrupted
if "judge_results" not in globals():
    judge_results = []

completed_judge_ids = {
    result["Question ID"] for result in judge_results
}

model = generation.model
tokenizer = generation.tokenizer

for _, row in evaluation_df.iterrows():

    q_id = row["Question ID"]

    # Skip questions already evaluated
    if q_id in completed_judge_ids:
        continue

    print(f"Evaluating {q_id}...")

    context = str(row["Retrieved Context"])[:9000]

    prompt = f"""
You are an evaluator for a Retrieval-Augmented Generation (RAG) system.

Evaluate the generated answer using the question, expected answer,
and retrieved document context.

QUESTION:
{row['Question']}

EXPECTED ANSWER:
{row['Expected Answer']}

GENERATED ANSWER:
{row['Generated Answer']}

RETRIEVED DOCUMENT CONTEXT:
{context}

Evaluate two metrics independently.

1. Correctness (1-5):
Compare the generated answer with the expected answer.
Consider meaning, factual accuracy, and completeness.
Do not require exact wording.

2. Faithfulness (1-5):
Check whether the generated answer is supported by the retrieved
document context.
Penalize unsupported claims and invented information.

Return ONLY valid JSON in this exact structure:

{{
    "correctness": 1,
    "faithfulness": 1,
    "reason": "Brief explanation of both scores"
}}

Scores must be integers between 1 and 5.
Do not give high scores merely because the answer sounds convincing.
"""

    messages = [
        {
            "role": "system",
            "content": "You are a strict and objective RAG evaluation judge."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[1]

    judge_answer = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    ).strip()

    # Extract JSON response
    try:
        match = re.search(r"\{.*\}", judge_answer, re.DOTALL)

        if not match:
            raise ValueError("JSON not found")

        scores = json.loads(match.group())

        correctness = int(scores["correctness"])
        faithfulness = int(scores["faithfulness"])

        if not (1 <= correctness <= 5 and 1 <= faithfulness <= 5):
            raise ValueError("Invalid score range")

        reason = scores.get("reason", "")

    except Exception:
        correctness = None
        faithfulness = None
        reason = "Could not parse judge response: " + judge_answer

    judge_results.append({
        "Question ID": q_id,
        "Correctness": correctness,
        "Faithfulness": faithfulness,
        "Evaluation Reason": reason
    })

    completed_judge_ids.add(q_id)

    print(
        f"Completed: {len(completed_judge_ids)}/20 | "
        f"Correctness: {correctness} | "
        f"Faithfulness: {faithfulness}"
    )

print("\nEvaluation process completed.")

# Create final evaluation DataFrame
judge_df = pd.DataFrame(judge_results)

final_evaluation_df = evaluation_df.merge(
    judge_df,
    on="Question ID",
    how="left"
)

display(
    final_evaluation_df[
        [
            "Question ID",
            "Correctness",
            "Faithfulness",
            "Evaluation Reason"
        ]
    ]
)


Evaluating Q1...
Completed: 1/20 | Correctness: 3 | Faithfulness: 4
Evaluating Q10...
Completed: 2/20 | Correctness: 4 | Faithfulness: 3
Evaluating Q11...
Completed: 3/20 | Correctness: 3 | Faithfulness: 3
Evaluating Q12...
Completed: 4/20 | Correctness: 4 | Faithfulness: 4
Evaluating Q13...
Completed: 5/20 | Correctness: 3 | Faithfulness: 3
Evaluating Q14...
Completed: 6/20 | Correctness: 4 | Faithfulness: 4
Evaluating Q15...
Completed: 7/20 | Correctness: 3 | Faithfulness: 3
Evaluating Q16...
Completed: 8/20 | Correctness: 3 | Faithfulness: 3
Evaluating Q17...
Completed: 9/20 | Correctness: 3 | Faithfulness: 3
Evaluating Q18...
Completed: 10/20 | Correctness: 3 | Faithfulness: 4
Evaluating Q19...
Completed: 11/20 | Correctness: 3 | Faithfulness: 3
Evaluating Q2...
Completed: 12/20 | Correctness: 3 | Faithfulness: 3
Evaluating Q20...
Completed: 13/20 | Correctness: 3 | Faithfulness: 4
Evaluating Q3...
Completed: 14/20 | Correctness: 3 | Faithfulness: 3
Evaluating Q4...
Completed: 15/2

,Question ID,Correctness,Faithfulness,Evaluation Reason
0,Q1,3,4,The generated answer closely matches the expec...
1,Q10,4,3,The generated answer closely matches the expec...
2,Q11,3,3,The generated answer correctly identifies deri...
3,Q12,4,4,The generated answer closely matches the expec...
4,Q13,3,3,The generated answer accurately describes the ...
5,Q14,4,4,The generated answer accurately describes the ...
6,Q15,3,3,The generated answer accurately describes the ...
7,Q16,3,3,The generated answer closely matches the expec...
8,Q17,3,3,The generated answer accurately describes the ...
9,Q18,3,4,The generated answer accurately describes the ...


In [60]:
import os
print(os.getcwd())

c:\Users\Nirmal\Desktop\document_qa_chatbot


In [61]:
import sys
from pathlib import Path

project_path = Path(
    r"C:\Users\Nirmal\Desktop\document_qa_chatbot"
)

src_path = str(project_path / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("Source path:", src_path)
print("Package exists:", (project_path / "src" / "document_qa_chatbot").exists())

Source path: C:\Users\Nirmal\Desktop\document_qa_chatbot\src
Package exists: True


In [62]:
import document_qa_chatbot.retrieval as retrieval
import document_qa_chatbot.generation as generation

print("Modules imported successfully!")

Modules imported successfully!


In [63]:
variables = [
    "chunks",
    "chunk_embeddings",
    "index",
    "bm25_index",
    "ground_truth",
    "evaluation_results",
    "evaluation_df",
    "judge_results",
    "final_evaluation_df"
]

for var in variables:
    print(f"{var}: {'Available' if var in globals() else 'Missing'}")

chunks: Available
chunk_embeddings: Available
index: Available
bm25_index: Available
ground_truth: Available
evaluation_results: Available
evaluation_df: Available
judge_results: Available
final_evaluation_df: Available


In [64]:
print("Judge evaluations:", len(judge_results))

display(pd.DataFrame(judge_results))

Judge evaluations: 20


,Question ID,Correctness,Faithfulness,Evaluation Reason
0,Q1,3,4,The generated answer closely matches the expec...
1,Q10,4,3,The generated answer closely matches the expec...
2,Q11,3,3,The generated answer correctly identifies deri...
3,Q12,4,4,The generated answer closely matches the expec...
4,Q13,3,3,The generated answer accurately describes the ...
5,Q14,4,4,The generated answer accurately describes the ...
6,Q15,3,3,The generated answer accurately describes the ...
7,Q16,3,3,The generated answer closely matches the expec...
8,Q17,3,3,The generated answer accurately describes the ...
9,Q18,3,4,The generated answer accurately describes the ...


In [65]:
from pathlib import Path
import pandas as pd

output_dir = Path(
    r"C:\Users\Nirmal\Desktop\document_qa_chatbot\data\evaluation"
)
output_dir.mkdir(parents=True, exist_ok=True)

# Save generated answers and retrieved context
evaluation_df.to_csv(
    output_dir / "evaluation_results.csv",
    index=False
)

# Save individual judge scores
pd.DataFrame(judge_results).to_csv(
    output_dir / "judge_results.csv",
    index=False
)

# Save final evaluation if available
if "final_evaluation_df" in globals():
    final_evaluation_df.to_csv(
        output_dir / "final_evaluation.csv",
        index=False
    )

print("Evaluation files saved successfully!")
print("Generated answers:", len(evaluation_df))
print("Judge evaluations:", len(judge_results))
print("Final evaluation available:", "final_evaluation_df" in globals())
print("Saved location:", output_dir)

Evaluation files saved successfully!
Generated answers: 20
Judge evaluations: 20
Final evaluation available: True
Saved location: C:\Users\Nirmal\Desktop\document_qa_chatbot\data\evaluation
